<a href="https://colab.research.google.com/github/jye110/ai-hw-spring-2026-aa/blob/main/Attack_MNIST_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# model.py
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28 -> 14
        x = self.pool(F.relu(self.conv2(x)))  # 14 -> 7
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [6]:
import random
import numpy as np
import torch

def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

def get_train_transform():
  return transforms.Compose([
      transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
      transforms.ToTensor(),
      transforms.Normalize((0.1307,), (0.3081,))
  ])


In [7]:
                                                                                                                                                                                                                                                                                        # train.py
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(42)

transform_train = get_train_transform()

train_data = datasets.MNIST("./data", train=True, download=True, transform=transform_train)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/5 - Loss: {avg_loss:.4f}")

  # torch.save(model.state_dict(), "mnist_cnn.pth")
  # print("Model saved as mnist_cnn.pth")


transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_data = datasets.MNIST("./data", train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%\n")

Epoch 1/5 - Loss: 0.2993
Epoch 2/5 - Loss: 0.1150
Epoch 3/5 - Loss: 0.0876
Epoch 4/5 - Loss: 0.0776
Epoch 5/5 - Loss: 0.0691
Test Accuracy: 99.28%



In [15]:
mean = 0.1307
std = 0.3081
lower = (0 - mean) / std
upper = (1 - mean) / std

# FGSM attack
def fgsm_attack(model, images, labels, epsilon):
    epsilon_norm = epsilon / std

    images = images.clone().detach().to(device)
    labels = labels.to(device)

    images.requires_grad = True

    outputs = model(images)
    loss = criterion(outputs, labels)

    model.zero_grad()
    loss.backward()

    adv_images = images + epsilon_norm * images.grad.sign()
    adv_images = torch.clamp(adv_images, lower, upper)

    return adv_images.detach()

# I-FGSM / PGD attack
def pgd_attack(model, images, labels, epsilon, alpha, steps):
    epsilon_norm = epsilon / std
    alpha_norm = alpha / std

    original_images = images.clone().detach()
    adv_images = original_images.clone().detach()

    for _ in range(steps):
        adv_images.requires_grad = True

        outputs = model(adv_images)
        loss = criterion(outputs, labels)

        model.zero_grad()
        loss.backward()

        adv_images = adv_images + alpha_norm * adv_images.grad.sign()

        perturbation = torch.clamp(
            adv_images - original_images,
            min=-epsilon_norm,
            max=epsilon_norm
        )

        adv_images = original_images + perturbation
        adv_images = torch.clamp(adv_images, lower, upper).detach()

    return adv_images

# Momentum I-FGSM attack
def mifgsm_attack(model, images, labels, epsilon, alpha, steps, decay=1.0):
    epsilon_norm = epsilon / std
    alpha_norm = alpha / std

    original_images = images.clone().detach()
    adv_images = original_images.clone().detach()

    momentum = torch.zeros_like(adv_images)

    for _ in range(steps):
        adv_images.requires_grad = True

        outputs = model(adv_images)
        loss = criterion(outputs, labels)

        model.zero_grad()
        loss.backward()

        grad = adv_images.grad

        # Normalize gradient for each image
        grad_norm = torch.mean(torch.abs(grad), dim=(1, 2, 3), keepdim=True)
        normalized_grad = grad / (grad_norm + 1e-8)

        # Add momentum
        momentum = decay * momentum + normalized_grad

        # Update adversarial image
        adv_images = adv_images + alpha_norm * momentum.sign()

        # Limit perturbation within epsilon-ball
        perturbation = torch.clamp(
            adv_images - original_images,
            min=-epsilon_norm,
            max=epsilon_norm
        )

        adv_images = original_images + perturbation

        # Clamp to valid normalized pixel range
        adv_images = torch.clamp(adv_images, lower, upper).detach()

    return adv_images

In [13]:
def evaluate_attack(model, loader, attack_fn, **attack_params):
    model.eval()

    originally_correct = 0
    fooled = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            clean_outputs = model(images)
            clean_preds = clean_outputs.argmax(dim=1)

        correct_mask = clean_preds == labels

        if correct_mask.sum().item() == 0:
            continue

        images_correct = images[correct_mask]
        labels_correct = labels[correct_mask]

        adv_images = attack_fn(model, images_correct, labels_correct, **attack_params)

        with torch.no_grad():
            adv_outputs = model(adv_images)
            adv_preds = adv_outputs.argmax(dim=1)

        originally_correct += labels_correct.size(0)
        fooled += (adv_preds != labels_correct).sum().item()

    asr = fooled / originally_correct
    return asr

In [17]:
epsilons = [0.05, 0.10, 0.20, 0.30]

for eps in epsilons:
    print(f"\nEpsilon = {eps}")

    fgsm_asr = evaluate_attack(
        model,
        test_loader,
        fgsm_attack,
        epsilon=eps
    )

    pgd_asr = evaluate_attack(
        model,
        test_loader,
        pgd_attack,
        epsilon=eps,
        alpha=eps / 10,
        steps=10
    )

    mifgsm_asr = evaluate_attack(
        model,
        test_loader,
        mifgsm_attack,
        epsilon=eps,
        alpha=eps / 10,
        steps=10,
        decay=1.0
    )

    print(f"FGSM ASR:     {fgsm_asr:.4f}")
    print(f"PGD ASR:      {pgd_asr:.4f}")
    print(f"MI-FGSM ASR:  {mifgsm_asr:.4f}")


Epsilon = 0.05
FGSM ASR:     0.0321
PGD ASR:      0.0504
MI-FGSM ASR:  0.0492

Epsilon = 0.1
FGSM ASR:     0.1354
PGD ASR:      0.3081
MI-FGSM ASR:  0.2882

Epsilon = 0.2
FGSM ASR:     0.6714
PGD ASR:      0.9590
MI-FGSM ASR:  0.9525

Epsilon = 0.3
FGSM ASR:     0.8708
PGD ASR:      0.9998
MI-FGSM ASR:  0.9995
